# 1. 📦 Imports & Setup

In [3]:
# Data manipulation
import pandas as pd

# Football data
from statsbombpy import sb

# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

# 2. 🔍 Explore Available Competitions

In [11]:
# Load all available competitions
competitions = sb.competitions()

# Filter La Liga (men's competitions only)
laliga = competitions[
    (competitions["competition_name"] == "La Liga") &
    (competitions["competition_gender"] == "male")
]

laliga[["competition_id", "season_id", "season_name"]]

,competition_id,season_id,season_name
38,11,90,2020/2021
39,11,42,2019/2020
40,11,4,2018/2019
41,11,1,2017/2018
42,11,2,2016/2017
43,11,27,2015/2016
44,11,26,2014/2015
45,11,25,2013/2014
46,11,24,2012/2013
47,11,23,2011/2012


# 3. ⚽ Load Matches

In [12]:
# Create list of (competition_id, season_id)
leagues = list(zip(laliga["competition_id"], laliga["season_id"]))

# Load matches across seasons
matches = pd.concat(
    [sb.matches(comp, season) for comp, season in leagues],
    ignore_index=True
)

print(f"Total matches: {len(matches)}")
matches.head()

Total matches: 868


,match_id,match_date,kick_off,competition,season,home_team,away_team,home_score,away_score,match_status,match_status_360,last_updated,last_updated_360,match_week,competition_stage,stadium,referee,home_managers,away_managers,data_version,shot_fidelity_version,xy_fidelity_version
0,3773386,2020-10-31,21:00:00.000,Spain - La Liga,2020/2021,Deportivo Alavés,Barcelona,1,1,available,available,2023-07-25T03:54:59.280826,2023-07-25T04:25:41.348202,8,Regular Season,Estadio de Mendizorroza,NaN,Pablo Javier Machín Díez,Ronald Koeman,1.1.0,2,2
1,3773565,2021-01-09,18:30:00.000,Spain - La Liga,2020/2021,Granada,Barcelona,0,4,available,available,2023-07-25T03:51:37.437064,2023-07-25T04:30:16.058384,18,Regular Season,Estadio Nuevo Los Cármenes,Ricardo De Burgos Bengoetxea,Diego Martínez Penas,Ronald Koeman,1.1.0,2,2
2,3773457,2021-05-16,18:30:00.000,Spain - La Liga,2020/2021,Barcelona,Celta Vigo,1,2,available,available,2022-12-02T09:26:39.496362,2023-04-27T23:03:53.506485,37,Regular Season,Spotify Camp Nou,NaN,Ronald Koeman,Eduardo Germán Coudet,1.1.0,2,2
3,3773631,2021-02-07,21:00:00.000,Spain - La Liga,2020/2021,Real Betis,Barcelona,2,3,available,available,2023-07-25T03:47:44.278651,2023-07-25T03:56:34.733180,22,Regular Season,Estadio Benito Villamarín,NaN,Manuel Luis Pellegrini Ripamonti,Ronald Koeman,1.1.0,2,2
4,3773665,2021-03-06,21:00:00.000,Spain - La Liga,2020/2021,Osasuna,Barcelona,0,2,available,available,2022-12-02T08:46:42.897589,2023-04-28T02:57:03.412841,26,Regular Season,Estadio El Sadar,Guillermo Cuadra Fernández,Jagoba Arrasate Elustondo,Ronald Koeman,1.1.0,2,2


# 4. 🎯 Select a Match & Load Lineups

In [13]:
# Example match
match_id = 3773386

# Inspect match info
matches[matches["match_id"] == match_id]

,match_id,match_date,kick_off,competition,season,home_team,away_team,home_score,away_score,match_status,match_status_360,last_updated,last_updated_360,match_week,competition_stage,stadium,referee,home_managers,away_managers,data_version,shot_fidelity_version,xy_fidelity_version
0,3773386,2020-10-31,21:00:00.000,Spain - La Liga,2020/2021,Deportivo Alavés,Barcelona,1,1,available,available,2023-07-25T03:54:59.280826,2023-07-25T04:25:41.348202,8,Regular Season,Estadio de Mendizorroza,NaN,Pablo Javier Machín Díez,Ronald Koeman,1.1.0,2,2


In [14]:
# Load lineups for the match
lineups = sb.lineups(match_id=match_id)

# Select a team
team = "Barcelona"
lineup = lineups[team]

lineup.head()

,player_id,player_name,player_nickname,jersey_number,country,cards,positions
0,4447,Martin Braithwaite Christensen,Martin Braithwaite,9,Denmark,[],"[{'position_id': 21, 'position': 'Left Wing', ..."
1,5203,Sergio Busquets i Burgos,Sergio Busquets,5,Spain,"[{'time': '43:12', 'card_type': 'Yellow Card',...","[{'position_id': 9, 'position': 'Right Defensi..."
2,5211,Jordi Alba Ramos,Jordi Alba,18,Spain,[],"[{'position_id': 6, 'position': 'Left Back', '..."
3,5213,Gerard Piqué Bernabéu,Gerard Piqué,3,Spain,[],"[{'position_id': 3, 'position': 'Right Center ..."
4,5477,Ousmane Dembélé,None,11,France,[],"[{'position_id': 17, 'position': 'Right Wing',..."


# 5. 🧹 Extract the Starting XI (Core Logic)

In [15]:
def get_starting_xi(match_id: int, team: str) -> pd.DataFrame:
    """
    Extract the starting XI for a given team and match.
    """

    lineup = sb.lineups(match_id=match_id)[team].copy()

    # Expand positions
    df = lineup.explode("positions")
    pos = pd.json_normalize(df["positions"])

    df = pd.concat(
        [df.drop(columns="positions").reset_index(drop=True), pos],
        axis=1
    )

    # Filter starters
    starters = df[df["start_reason"] == "Starting XI"].copy()

    # Clean player names
    starters["player"] = (
        starters["player_nickname"]
        .fillna(starters["player_name"])
        .str.strip()
    )

    # Select relevant columns
    starters = starters[[
        "player",
        "jersey_number",
        "position"
    ]].sort_values("jersey_number")

    return starters.reset_index(drop=True)

In [16]:
starting_xi = get_starting_xi(match_id, team)
starting_xi

,player,jersey_number,position
0,Gerard Piqué,3,Right Center Back
1,Sergio Busquets,5,Right Defensive Midfield
2,Antoine Griezmann,7,Center Forward
3,Lionel Messi,10,Center Attacking Midfield
4,Ousmane Dembélé,11,Right Wing
5,Neto,13,Goalkeeper
6,Clément Lenglet,15,Left Center Back
7,Jordi Alba,18,Left Back
8,Sergi Roberto,20,Right Back
9,Frenkie de Jong,21,Left Defensive Midfield


# 9. 🧩 Simple Formation Detection

In [24]:
def infer_formation(df: pd.DataFrame) -> str:
    """
    Infer formation from starting XI using position groupings.
    """

    # Remove goalkeeper
    df = df[df["position"] != "Goalkeeper"].copy()

    # Remove duplicates (safety)
    df = df.drop_duplicates(subset="player")

    # Define position groups
    defence_positions = [
        "Right Back", "Left Back",
        "Right Center Back", "Left Center Back",
        "Center Back"
    ]

    midfield_positions = [
        "Right Defensive Midfield", "Left Defensive Midfield",
        "Center Defensive Midfield",
        "Center Midfield",
        "Right Midfield", "Left Midfield",
        "Center Attacking Midfield"
    ]

    attack_positions = [
        "Right Wing", "Left Wing",
        "Center Forward", "Striker"
    ]

    # Count players per line
    defence = df["position"].isin(defence_positions).sum()
    midfield = df["position"].isin(midfield_positions).sum()
    attack = df["position"].isin(attack_positions).sum()

    return f"{defence}-{midfield}-{attack}"

In [25]:
formation = infer_formation(starting_xi)
print("Inferred formation:", formation)

Inferred formation: 4-3-3


In [26]:
def get_starting_xi(match_id: int, team: str) -> pd.DataFrame:
    """
    Extract a clean starting XI table.
    """

    lineup = sb.lineups(match_id=match_id)[team]

    df = (
        lineup
        .explode("positions")
        .pipe(lambda d: pd.concat(
            [d.drop(columns="positions").reset_index(drop=True),
             pd.json_normalize(d["positions"])],
            axis=1
        ))
    )

    starters = (
        df[df["start_reason"] == "Starting XI"]
        .assign(
            player=lambda d: d["player_nickname"]
            .fillna(d["player_name"])
            .str.strip()
        )
        [["player", "jersey_number", "position"]]
        .drop_duplicates(subset="player")
        .sort_values("jersey_number")
        .reset_index(drop=True)
    )

    return starters